# 03 - Fine-tune Qwen2-VL-2B with QLoRA (Week 2)

**What this notebook does**
1. Installs Unsloth and loads `splits.csv` from the split notebook.
2. Builds training examples from `vlm_train` (+ some extra good images), each one an image, a question and a JSON answer.
3. Asks the *untrained* model about 3 eval images (the "before" picture).
4. Trains small LoRA adapters on one T4.
5. Asks the *trained* model about 10 eval images (the "after" picture). This is a sanity check, not the real evaluation (Week 3).
6. Saves the adapters.

**Settings needed (right panel → Session options):** Accelerator **GPU T4 x2** (Unsloth uses one) · Internet **On** · Inputs: **MVTec AD** dataset and your **split notebook's output**.

**What the model learns to answer:** `{"defective": true, "defect_type": "scratch_head", "location": "top"}`. The location is a coarse 3×3 grid position computed from each defect's mask. Confidence is **not** trained as text; in Week 3 we read it from the model's probability of answering `true`, which is what AUROC needs.


## Step 1 - Install Unsloth (2-4 minutes)

In [ ]:
%%capture
!pip install -q unsloth

## Step 2 - Check the GPU

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth trains on one GPU
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 in Session options"
print(torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Step 3 - Load the split
Finds `splits.csv` from your split notebook and points every path at this session's dataset folder.

In [ ]:
import pandas as pd
from pathlib import Path

split_files = list(Path('/kaggle/input').rglob('splits.csv'))
assert split_files, "splits.csv not found: Add Input -> Your Work -> your split notebook"
df = pd.read_csv(split_files[0])

roots = [p.parent for p in Path('/kaggle/input').rglob('bottle') if p.is_dir() and (p / 'train').exists()]
assert roots, "MVTec AD not found: Add Input -> the MVTec AD dataset"
ROOT = roots[0]

def rebase(p):
    if not isinstance(p, str):
        return p
    parts = p.split('/')
    if 'ground_truth' in parts:
        i = parts.index('ground_truth') - 1
    else:
        i = len(parts) - 4
    return str(ROOT.joinpath(*parts[i:]))

df['path'] = df['path'].map(rebase)
df['mask_path'] = df['mask_path'].map(rebase)
assert Path(df.path.iloc[0]).exists() and Path(df.mask_path.dropna().iloc[0]).exists(), "paths don't resolve"
print(len(df), 'rows |', df.role.value_counts().to_dict())

## Step 4 - Helper functions
Prompt, answer, image loading and JSON parsing. No training happens here.

In [ ]:
import json, random
from pathlib import Path
import numpy as np
from PIL import Image

IMG_SIZE = 448          # 448 = 16 x 28 -> about 256 visual tokens per image
GRID = [["top-left", "top", "top-right"],
        ["left", "center", "right"],
        ["bottom-left", "bottom", "bottom-right"]]

def location_from_mask(mask_path):
    """Coarse 3x3 location of the defect, from the centre of its mask."""
    m = np.array(Image.open(mask_path).convert("L")) > 0
    if not m.any():
        return "none"
    ys, xs = np.nonzero(m)
    h, w = m.shape
    row = min(int(ys.mean() / h * 3), 2)
    col = min(int(xs.mean() / w * 3), 2)
    return GRID[row][col]

def make_prompt(category, allowed_types):
    name = category.replace("_", " ")
    types = ", ".join(sorted(allowed_types))
    return (f"You are inspecting a {name} on a factory line. Is it defective? "
            f"Possible defect types: {types}. "
            'Answer only with JSON: {"defective": true/false, "defect_type": "<type or none>", '
            '"location": "<top-left|top|top-right|left|center|right|bottom-left|bottom|bottom-right|none>"}')

def make_answer(row):
    if row["label"] == "good":
        ans = {"defective": False, "defect_type": "none", "location": "none"}
    else:
        ans = {"defective": True, "defect_type": row["defect_type"],
               "location": location_from_mask(row["mask_path"])}
    return json.dumps(ans)

def load_image(path):
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)

def to_conversation(image, prompt, answer):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": image},
                                     {"type": "text", "text": prompt}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]}

def parse_json(text):
    """Pull the first {...} out of the model output; None if it isn't valid JSON."""
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        return json.loads(text[start:end])
    except ValueError:
        return None


## Step 5 - Build the training set
All of `vlm_train` (631 defect + 231 good) plus ~27 good images per category from `train_good`, so good and defect are roughly balanced. Then a leakage check.

In [ ]:
random.seed(42)
allowed = df[df.label == 'defect'].groupby('category').defect_type.unique().to_dict()

vlm = df[df.role == 'vlm_train']
extra_good = (df[df.role == 'train_good']
              .groupby('category', group_keys=False)
              .apply(lambda g: g.sample(min(27, len(g)), random_state=42)))
train_rows = pd.concat([vlm, extra_good]).sample(frac=1, random_state=42).reset_index(drop=True)

eval_paths = set(df.loc[df.role == 'eval', 'path'])
assert not (set(train_rows.path) & eval_paths), "LEAK: a training image is in eval"
print('training examples:', len(train_rows), '|', train_rows.label.value_counts().to_dict())
print('leakage check passed ✓')

In [ ]:
from tqdm.auto import tqdm
train_rows['prompt'] = [make_prompt(c, allowed[c]) for c in train_rows.category]
train_rows['answer'] = [make_answer(r) for _, r in tqdm(train_rows.iterrows(), total=len(train_rows), desc='answers')]
train_data = [to_conversation(load_image(p), q, a)
              for p, q, a in tqdm(zip(train_rows.path, train_rows.prompt, train_rows.answer), total=len(train_rows), desc='images')]
train_rows[['path', 'category', 'label', 'defect_type', 'answer']].to_csv('/kaggle/working/train_examples.csv', index=False)
print(train_rows.prompt.iloc[0]); print('->', train_rows.answer.iloc[0])

## Step 6 - Load the model (4-bit) and add LoRA adapters

In [ ]:
from unsloth import FastVisionModel, is_bf16_supported

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    random_state=3407,
)
print('GPU memory used:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## Step 7 - Before training: ask the untrained model
Expect messy or wrong answers. That's the point of the comparison.

In [ ]:
def ask(row, max_new_tokens=64):
    FastVisionModel.for_inference(model)
    image = load_image(row['path'])
    messages = [{"role": "user", "content": [{"type": "image"},
                {"type": "text", "text": make_prompt(row['category'], allowed[row['category']])}]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

eval_rows = df[df.role == 'eval'].sample(10, random_state=1).reset_index(drop=True)
for _, r in eval_rows.head(3).iterrows():
    print(f"[{r.category} | truth: {r.defect_type}]\n  {ask(r)}\n")

## Step 8 - Train
One pass over the training set (~158 steps). Watch `loss` go down. Time on a T4 is not yet measured: note it for the README.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import inspect, time

cfg = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="/kaggle/working/outputs",
    report_to="none",
    save_strategy="no",
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)
# TRL renamed max_seq_length -> max_length; use whichever this version has
key = "max_length" if "max_length" in inspect.signature(SFTConfig.__init__).parameters else "max_seq_length"
cfg[key] = 1024

FastVisionModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                     data_collator=UnslothVisionDataCollator(model, tokenizer),
                     train_dataset=train_data, args=SFTConfig(**cfg))
t0 = time.time()
stats = trainer.train()
print(f"training time: {(time.time() - t0) / 60:.1f} min | final loss: {stats.training_loss:.4f}")
print('peak GPU memory:', round(torch.cuda.max_memory_allocated() / 1e9, 2), 'GB')

## Step 9 - After training: ask the trained model about 10 eval images
A sanity check only. The real scores (AUROC, F1 per category) come in Week 3.

In [ ]:
results = []
for _, r in eval_rows.iterrows():
    raw = ask(r)
    pred = parse_json(raw) or {}
    results.append(dict(category=r.category, truth=r.defect_type, raw=raw,
                        valid_json=bool(pred),
                        defective_ok=pred.get('defective') == (r.label == 'defect'),
                        type_ok=pred.get('defect_type') == (r.defect_type if r.label == 'defect' else 'none')))
res = pd.DataFrame(results)
print('valid JSON:', res.valid_json.sum(), '/ 10 | defective correct:', res.defective_ok.sum(),
      '/ 10 | type correct:', res.type_ok.sum(), '/ 10')
res[['category', 'truth', 'raw']]

## Step 10 - Save the adapters
Then **Save Version → Save & Run All (Commit)** so the adapters become a permanent output for Week 3.

In [ ]:
out_dir = '/kaggle/working/qwen2vl-mvtec-lora'
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print('saved to', out_dir)
!du -sh {out_dir}